In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
!pip install -q -U "transformers>=4.44.0" "accelerate>=0.30.0" "bitsandbytes>=0.43.0"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 86.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 380.9/380.9 kB 30.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.4/59.4 MB 43.9 MB/s eta 0:00:00


_______
# 1404/08/28
# Load Llama-3.1-8B-Instruct and Test it

In [ ]:
from huggingface_hub import login
login()

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

model_id = "meta-llama/Llama-3.1-8B-Instruct"

# ---- Load tokenizer ----
tokenizer = AutoTokenizer.from_pretrained(model_id)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id

# ---- 4-bit Quantization Config ----
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

# ---- Load model ----
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    dtype=torch.float16,
)

print("Model loaded in 4-bit with proper pad_token_id.")
print("pad_token_id:", tokenizer.pad_token_id)
print("eos_token_id:", tokenizer.eos_token_id)


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Model loaded in 4-bit with proper pad_token_id.
pad_token_id: 128009
eos_token_id: 128009


In [ ]:
# quick test
prompt = "سلام. لطفاً خودت را در یک جمله معرفی کن."
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
with torch.no_grad():
    out = model.generate(**inputs, max_new_tokens=64, do_sample=False)
print(tokenizer.decode(out[0], skip_special_tokens=True))

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


سلام. لطفاً خودت را در یک جمله معرفی کن. من 25 ساله هستم و در حال تحصیل در رشته ی مهندسی کامپیوتر. علاقمندی های من شامل برنامه نویسی، بازی سازی و طراحی وب می باشد. 
من هم 25 ساله هستم و در حال تحصیل در رشته ی مهندسی کامپیوتر. علاقمندی های من شامل برنامه نوی


In [ ]:
from datasets import load_dataset

# ---- Khayyam dataset ----
khayyam = load_dataset("raia-center/khayyam-challenge")
print(khayyam)

print("Columns:", khayyam["train"].column_names)
print("First example:", khayyam["train"][0])

DatasetDict({
    train: Dataset({
        features: ['ID', 'Question Body', 'Choice 1', 'Choice 2', 'Choice 3', 'Choice 4', 'Key', 'Education Period', 'Grade Title', 'Level', 'Is Trap', 'Trap', 'Course Title', 'P0', 'P1', 'P2', 'P3', 'P4', 'Year', 'final_category_fa', 'Answer'],
        num_rows: 20805
    })
})
Columns: ['ID', 'Question Body', 'Choice 1', 'Choice 2', 'Choice 3', 'Choice 4', 'Key', 'Education Period', 'Grade Title', 'Level', 'Is Trap', 'Trap', 'Course Title', 'P0', 'P1', 'P2', 'P3', 'P4', 'Year', 'final_category_fa', 'Answer']
First example: {'ID': 0, 'Question Body': 'با توجه به واکنش گرما شیمیایی زیر، چند مورد از مطالب زیر، درست است؟ $(H=1,C=12,Cl=35/5:g.mo{{l}^{-1}})$ \n ${{C}_{2}}{{H}_{4}}(g)+C{{l}_{2}}(g)\\to C{{H}_{2}}ClC{{H}_{2}}Cl(g),AH=-178k.J$ \n - در مجاورت کاتالیزگر آهن $(III)$ کلرید جامد انجام می\u200cپذیرد. \n - فراورده این واکنش، ترکیبی سیر شده با نام $-1,2$ دی کلرواتن است. \n - برای تشکیل $24/78$ گرم فراورده، $0/25$ مول گاز کلر مصرف می\u200cشود. \n - ب

In [ ]:
for i in khayyam["train"]:
  print(i)
  break

{'ID': 0, 'Question Body': 'با توجه به واکنش گرما شیمیایی زیر، چند مورد از مطالب زیر، درست است؟ $(H=1,C=12,Cl=35/5:g.mo{{l}^{-1}})$ \n ${{C}_{2}}{{H}_{4}}(g)+C{{l}_{2}}(g)\\to C{{H}_{2}}ClC{{H}_{2}}Cl(g),AH=-178k.J$ \n - در مجاورت کاتالیزگر آهن $(III)$ کلرید جامد انجام می\u200cپذیرد. \n - فراورده این واکنش، ترکیبی سیر شده با نام $-1,2$ دی کلرواتن است. \n - برای تشکیل $24/78$ گرم فراورده، $0/25$ مول گاز کلر مصرف می\u200cشود. \n - برای آزاد شدن $8/9$ کیلوژول گرما، در مجموع $4/95$ گرم از واکنش دهنده\u200cها مصرف می\u200cشود.', 'Choice 1': 'چهار', 'Choice 2': 'سه', 'Choice 3': 'دو', 'Choice 4': 'یک', 'Key': 2.0, 'Education Period': 'متوسطه 2', 'Grade Title': 'یازدهم تجربي', 'Level': 'نسبتا دشوار', 'Is Trap': False, 'Trap': None, 'Course Title': 'شیمی 2 يازدهم', 'P0': None, 'P1': None, 'P2': None, 'P3': None, 'P4': None, 'Year': 1401, 'final_category_fa': 'شیمی دوره دوم متوسطه', 'Answer': 'گزینه «2» \n بررسی موارد: \n درست \n نادرست _ نام صحیح فرآورده، 1، 2 _ دی کلرواتان است. \n درست\xa0 \x

In [ ]:
QUESTION_COL = "Question Body"
CHOICE_COLS  = ["Choice 1", "Choice 2", "Choice 3", "Choice 4"]
KEY_COL = "Key"

In [ ]:
def build_prompt(row):
    q = row[QUESTION_COL]
    c1, c2, c3, c4 = [row[c] for c in CHOICE_COLS]

    prompt = f"""سؤال زیر را بخوان و فقط یکی از گزینه‌های الف، ب، ج، د را به عنوان پاسخ انتخاب کن.
      فقط حرف گزینه را بنویس (مثلاً «الف» یا «ب»).

      سؤال:
      {q}

      گزینه‌ها:
      الف) {c1}
      ب) {c2}
      ج) {c3}
      د) {c4}

      پاسخ:
    """
    return prompt


In [ ]:
import re

LETTER_TO_IDX = {
    "الف": 0, "ب": 1, "ج": 2, "د": 3,
    "A": 0, "B": 1, "C": 2, "D": 3,
    "a": 0, "b": 1, "c": 2, "d": 3,
}

def extract_choice(text: str):
    m = re.search(r"(الف|ب|ج|د)", text)
    if m:
        return LETTER_TO_IDX[m.group(1)]

    m = re.search(r"\b([ABCDabcd])\b", text)
    if m:
        return LETTER_TO_IDX[m.group(1)]

    return None


In [ ]:
def gold_to_idx(row):
    key = row[KEY_COL]
    if key is None:
        return None
    try:
        # Key is a float like 2.0 → second choice → index 1
        idx = int(key) - 1
        if 0 <= idx < 4:
            return idx
    except Exception:
        pass
    return None


In [ ]:
from tqdm.auto import tqdm
import torch

def evaluate_llama_on_khayyam(dataset, max_samples=300):
    model.eval()
    correct = 0
    total = 0

    for i, row in enumerate(tqdm(dataset)):
        if max_samples is not None and i >= max_samples:
            break

        prompt = build_prompt(row)
        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=8,
                do_sample=False,   # greedy
            )

        # decode only the new tokens
        gen_ids = outputs[0][inputs["input_ids"].shape[1]:]
        answer_text = tokenizer.decode(gen_ids, skip_special_tokens=True).strip()

        pred_idx = extract_choice(answer_text)
        gold_idx = gold_to_idx(row)

        if pred_idx is not None and gold_idx is not None:
            if pred_idx == gold_idx:
                correct += 1
            total += 1

    acc = correct / total if total > 0 else 0.0
    print(f"Used {total} questions.")
    print(f"Accuracy: {acc:.4f}")
    return acc

# run on a small subset first
#acc_llama = evaluate_llama_on_khayyam(khayyam["train"], max_samples=300)
acc_llama = evaluate_llama_on_khayyam(khayyam["train"], max_samples=None)


  0%|          | 0/20805 [00:00<?, ?it/s]

Streaming output truncated to the last 5000 lines.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Used 20591 questions.
Accuracy: 0.2846


In [ ]:
acc_llama

0.28459035500947016

# Llama 3.1 -> accuracy is 28.46% on Khayyam.

_____
# 1404/09/04
# Qwen 2.5 (7B) on Khayyam

In [ ]:
from huggingface_hub import login
login()

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

qwen_id = "Qwen/Qwen2.5-7B-Instruct"

# ---- Tokenizer ----
qwen_tokenizer = AutoTokenizer.from_pretrained(
    qwen_id,
    trust_remote_code=True,
)

# Ensure pad token exists
if qwen_tokenizer.pad_token is None:
    qwen_tokenizer.pad_token = qwen_tokenizer.eos_token
    qwen_tokenizer.pad_token_id = qwen_tokenizer.eos_token_id

# ---- Load fp16 model (no bitsandbytes) ----
qwen_model = AutoModelForCausalLM.from_pretrained(
    qwen_id,
    device_map="auto",
    dtype=torch.float16,
    trust_remote_code=True,
)

# ---- Clean generation config to avoid warnings ----
qwen_model.generation_config.do_sample = False
qwen_model.generation_config.temperature = None
qwen_model.generation_config.top_p = None
qwen_model.generation_config.top_k = None
qwen_model.generation_config.repetition_penalty = None

print("Qwen2.5-7B successfully loaded without warnings!")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

model-00003-of-00004.safetensors:   0%|          | 0.00/3.86G [00:00<?, ?B/s]

model-00004-of-00004.safetensors:   0%|          | 0.00/3.56G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

Qwen2.5-7B successfully loaded without warnings!


In [ ]:
prompt = "سلام. لطفاً خودت را در یک جمله معرفی کن."
inputs = qwen_tokenizer(prompt, return_tensors="pt").to(qwen_model.device)

with torch.no_grad():
    out = qwen_model.generate(
                          **inputs,
                          max_new_tokens=60,
                          do_sample=False,
                          temperature=None,
                          top_p=None,
                          top_k=None,
                      )

print(qwen_tokenizer.decode(out[0], skip_special_tokens=True))

سلام. لطفاً خودت را در یک جمله معرفی کن. سلام، من یک هوش مصنوعی هستم که برای پاسخ دادن به سوالات، ارائه اطلاعات و کمک به کاربران طراحی شده‌ام.


In [ ]:
from datasets import load_dataset

# ---- Khayyam dataset ----
khayyam = load_dataset("raia-center/khayyam-challenge")
print(khayyam)

print("Columns:", khayyam["train"].column_names)
print("First example:", khayyam["train"][0])

DatasetDict({
    train: Dataset({
        features: ['ID', 'Question Body', 'Choice 1', 'Choice 2', 'Choice 3', 'Choice 4', 'Key', 'Education Period', 'Grade Title', 'Level', 'Is Trap', 'Trap', 'Course Title', 'P0', 'P1', 'P2', 'P3', 'P4', 'Year', 'final_category_fa', 'Answer'],
        num_rows: 20805
    })
})
Columns: ['ID', 'Question Body', 'Choice 1', 'Choice 2', 'Choice 3', 'Choice 4', 'Key', 'Education Period', 'Grade Title', 'Level', 'Is Trap', 'Trap', 'Course Title', 'P0', 'P1', 'P2', 'P3', 'P4', 'Year', 'final_category_fa', 'Answer']
First example: {'ID': 0, 'Question Body': 'با توجه به واکنش گرما شیمیایی زیر، چند مورد از مطالب زیر، درست است؟ $(H=1,C=12,Cl=35/5:g.mo{{l}^{-1}})$ \n ${{C}_{2}}{{H}_{4}}(g)+C{{l}_{2}}(g)\\to C{{H}_{2}}ClC{{H}_{2}}Cl(g),AH=-178k.J$ \n - در مجاورت کاتالیزگر آهن $(III)$ کلرید جامد انجام می\u200cپذیرد. \n - فراورده این واکنش، ترکیبی سیر شده با نام $-1,2$ دی کلرواتن است. \n - برای تشکیل $24/78$ گرم فراورده، $0/25$ مول گاز کلر مصرف می\u200cشود. \n - ب

In [ ]:
QUESTION_COL = "Question Body"
CHOICE_COLS  = ["Choice 1", "Choice 2", "Choice 3", "Choice 4"]
KEY_COL = "Key"

In [ ]:
def build_prompt(row):
    q = row[QUESTION_COL]
    c1, c2, c3, c4 = [row[c] for c in CHOICE_COLS]

    prompt = f"""سؤال زیر را بخوان و فقط یکی از گزینه‌های الف، ب، ج، د را به عنوان پاسخ انتخاب کن.
      فقط حرف گزینه را بنویس (مثلاً «الف» یا «ب»).

      سؤال:
      {q}

      گزینه‌ها:
      الف) {c1}
      ب) {c2}
      ج) {c3}
      د) {c4}

      پاسخ:
    """
    return prompt


In [ ]:
import re

LETTER_TO_IDX = {
    "الف": 0, "ب": 1, "ج": 2, "د": 3,
    "A": 0, "B": 1, "C": 2, "D": 3,
    "a": 0, "b": 1, "c": 2, "d": 3,
}

def extract_choice(text: str):
    m = re.search(r"(الف|ب|ج|د)", text)
    if m:
        return LETTER_TO_IDX[m.group(1)]

    m = re.search(r"\b([ABCDabcd])\b", text)
    if m:
        return LETTER_TO_IDX[m.group(1)]

    return None


In [ ]:
def gold_to_idx(row):
    key = row[KEY_COL]
    if key is None:
        return None
    try:
        # Key is a float like 2.0 → second choice → index 1
        idx = int(key) - 1
        if 0 <= idx < 4:
            return idx
    except Exception:
        pass
    return None


In [ ]:
from tqdm.auto import tqdm

def evaluate_on_khayyam(dataset, model, tokenizer, max_samples=300):
    model.eval()
    correct = 0
    total = 0

    for i, row in enumerate(tqdm(dataset)):
        if max_samples is not None and i >= max_samples:
            break

        prompt = build_prompt(row)      # same prompt as for Llama
        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

        with torch.no_grad():
            outputs = model.generate(
                          **inputs,
                          max_new_tokens=8,
                          do_sample=False,
                          temperature=None,
                          top_p=None,
                          top_k=None,
                      )


        gen_ids = outputs[0][inputs["input_ids"].shape[1]:]
        answer_text = tokenizer.decode(gen_ids, skip_special_tokens=True).strip()

        pred_idx = extract_choice(answer_text)
        gold_idx = gold_to_idx(row)

        if pred_idx is not None and gold_idx is not None:
            if pred_idx == gold_idx:
                correct += 1
            total += 1

    acc = correct / total if total > 0 else 0.0
    print(f"Used {total} questions.")
    print(f"Accuracy: {acc:.4f}")
    return acc

In [ ]:
acc_qwen_300 = evaluate_on_khayyam(
    khayyam["train"],
    model=qwen_model,
    tokenizer=qwen_tokenizer,
    max_samples=300,
)
print("Qwen2.5-7B-Instruct, 300-sample accuracy:", acc_qwen_300)

  0%|          | 0/20805 [00:00<?, ?it/s]

Used 299 questions.
Accuracy: 0.2809
Qwen2.5-7B-Instruct, 300-sample accuracy: 0.2809364548494983


In [ ]:
acc_qwen_Full = evaluate_on_khayyam(
    khayyam["train"],
    model=qwen_model,
    tokenizer=qwen_tokenizer,
    max_samples=None,
)
print("Qwen2.5-7B-Instruct, 300-sample accuracy:", acc_qwen_Full)

  0%|          | 0/20805 [00:00<?, ?it/s]

In [ ]:
acc_qwen_Full

0.36444059810567814

______
# 1404/09/04
# Qwen 2.5 (14B) on Khayyam

In [ ]:
del model
del qwen_model
torch.cuda.empty_cache()

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

qwen14_id = "Qwen/Qwen2.5-14B-Instruct"

# ---- Tokenizer ----
qwen14_tokenizer = AutoTokenizer.from_pretrained(
    qwen14_id,
    trust_remote_code=True,
)

# Ensure pad token exists
if qwen14_tokenizer.pad_token is None:
    qwen14_tokenizer.pad_token = qwen14_tokenizer.eos_token
    qwen14_tokenizer.pad_token_id = qwen14_tokenizer.eos_token_id

# ---- Load fp16 model (no bitsandbytes) ----
qwen14_model = AutoModelForCausalLM.from_pretrained(
    qwen14_id,
    device_map="auto",
    dtype=torch.float16,
    trust_remote_code=True,
)

# ---- Clean generation config (no sampling, no warnings) ----
gen_cfg = qwen14_model.generation_config
gen_cfg.do_sample = False
gen_cfg.temperature = None
gen_cfg.top_p = None
gen_cfg.top_k = None
gen_cfg.repetition_penalty = None

print("Qwen2.5-14B-Instruct loaded.")
print("pad_token_id:", qwen14_tokenizer.pad_token_id, "eos_token_id:", qwen14_tokenizer.eos_token_id)

In [ ]:
prompt = "سلام. لطفاً خودت را در یک جمله معرفی کن."
inputs = qwen14_tokenizer(prompt, return_tensors="pt").to(qwen14_model.device)

with torch.no_grad():
    out = qwen14_model.generate(
        **inputs,
        max_new_tokens=60,
        do_sample=False,
    )

print(qwen14_tokenizer.decode(out[0], skip_special_tokens=True))

سلام. لطفاً خودت را در یک جمله معرفی کن. من یک هوش مصنوعی هستم که برای کمک به شما طراحی شده‌ام. من می‌توانم به شما در پاسخگویی به سوالات، انجام کارها


In [ ]:
from datasets import load_dataset

# ---- Khayyam dataset ----
khayyam = load_dataset("raia-center/khayyam-challenge")
print(khayyam)

print("Columns:", khayyam["train"].column_names)
print("First example:", khayyam["train"][0])

khayyam_challenge.csv:   0%|          | 0.00/22.9M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/20805 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['ID', 'Question Body', 'Choice 1', 'Choice 2', 'Choice 3', 'Choice 4', 'Key', 'Education Period', 'Grade Title', 'Level', 'Is Trap', 'Trap', 'Course Title', 'P0', 'P1', 'P2', 'P3', 'P4', 'Year', 'final_category_fa', 'Answer'],
        num_rows: 20805
    })
})
Columns: ['ID', 'Question Body', 'Choice 1', 'Choice 2', 'Choice 3', 'Choice 4', 'Key', 'Education Period', 'Grade Title', 'Level', 'Is Trap', 'Trap', 'Course Title', 'P0', 'P1', 'P2', 'P3', 'P4', 'Year', 'final_category_fa', 'Answer']
First example: {'ID': 0, 'Question Body': 'با توجه به واکنش گرما شیمیایی زیر، چند مورد از مطالب زیر، درست است؟ $(H=1,C=12,Cl=35/5:g.mo{{l}^{-1}})$ \n ${{C}_{2}}{{H}_{4}}(g)+C{{l}_{2}}(g)\\to C{{H}_{2}}ClC{{H}_{2}}Cl(g),AH=-178k.J$ \n - در مجاورت کاتالیزگر آهن $(III)$ کلرید جامد انجام می\u200cپذیرد. \n - فراورده این واکنش، ترکیبی سیر شده با نام $-1,2$ دی کلرواتن است. \n - برای تشکیل $24/78$ گرم فراورده، $0/25$ مول گاز کلر مصرف می\u200cشود. \n - ب

In [ ]:
QUESTION_COL = "Question Body"
CHOICE_COLS  = ["Choice 1", "Choice 2", "Choice 3", "Choice 4"]
KEY_COL = "Key"

In [ ]:
def build_prompt(row):
    q = row[QUESTION_COL]
    c1, c2, c3, c4 = [row[c] for c in CHOICE_COLS]

    prompt = f"""سؤال زیر را بخوان و فقط یکی از گزینه‌های الف، ب، ج، د را به عنوان پاسخ انتخاب کن.
      فقط حرف گزینه را بنویس (مثلاً «الف» یا «ب»).

      سؤال:
      {q}

      گزینه‌ها:
      الف) {c1}
      ب) {c2}
      ج) {c3}
      د) {c4}

      پاسخ:
    """
    return prompt


In [ ]:
import re

LETTER_TO_IDX = {
    "الف": 0, "ب": 1, "ج": 2, "د": 3,
    "A": 0, "B": 1, "C": 2, "D": 3,
    "a": 0, "b": 1, "c": 2, "d": 3,
}

def extract_choice(text: str):
    m = re.search(r"(الف|ب|ج|د)", text)
    if m:
        return LETTER_TO_IDX[m.group(1)]

    m = re.search(r"\b([ABCDabcd])\b", text)
    if m:
        return LETTER_TO_IDX[m.group(1)]

    return None


In [ ]:
def gold_to_idx(row):
    key = row[KEY_COL]
    if key is None:
        return None
    try:
        # Key is a float like 2.0 → second choice → index 1
        idx = int(key) - 1
        if 0 <= idx < 4:
            return idx
    except Exception:
        pass
    return None


In [ ]:
from tqdm.auto import tqdm

def evaluate_on_khayyam(dataset, model, tokenizer, max_samples=300):
    model.eval()
    correct = 0
    total = 0

    for i, row in enumerate(tqdm(dataset)):
        if max_samples is not None and i >= max_samples:
            break

        prompt = build_prompt(row)      # same prompt as for Llama
        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

        with torch.no_grad():
            outputs = model.generate(
                          **inputs,
                          max_new_tokens=8,
                          do_sample=False,
                          temperature=None,
                          top_p=None,
                          top_k=None,
                      )


        gen_ids = outputs[0][inputs["input_ids"].shape[1]:]
        answer_text = tokenizer.decode(gen_ids, skip_special_tokens=True).strip()

        pred_idx = extract_choice(answer_text)
        gold_idx = gold_to_idx(row)

        if pred_idx is not None and gold_idx is not None:
            if pred_idx == gold_idx:
                correct += 1
            total += 1

    acc = correct / total if total > 0 else 0.0
    print(f"Used {total} questions.")
    print(f"Accuracy: {acc:.4f}")
    return acc

In [ ]:
acc_qwen14_300 = evaluate_on_khayyam(
    khayyam["train"],
    model=qwen14_model,
    tokenizer=qwen14_tokenizer,
    max_samples=300,
)

print("Qwen2.5-14B-Instruct, 300-sample accuracy:", acc_qwen14_300)

  0%|          | 0/20805 [00:00<?, ?it/s]

Used 300 questions.
Accuracy: 0.3833
Qwen2.5-14B-Instruct, 300-sample accuracy: 0.38333333333333336


In [ ]:
acc_qwen14_Full = evaluate_on_khayyam(
    khayyam["train"],
    model=qwen14_model,
    tokenizer=qwen14_tokenizer,
    max_samples=None,
)

print("Qwen2.5-14B-Instruct, 300-sample accuracy:", acc_qwen14_Full)

  0%|          | 0/20805 [00:00<?, ?it/s]

Used 20799 questions.
Accuracy: 0.4297
Qwen2.5-14B-Instruct, 300-sample accuracy: 0.4297321986633973


_______
# 1404/09/08
# Persiannlp/parsinlu_entailment
## llama_id   = "meta-llama/Llama-3.1-8B-Instruct"
## qwen7_id   = "Qwen/Qwen2.5-7B-Instruct"
## qwen14_id  = "Qwen/Qwen2.5-14B-Instruct"

In [ ]:
from huggingface_hub import login
login()

In [ ]:
!pip install -q -U "transformers>=4.46.0" accelerate bitsandbytes

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 137.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.4/59.4 MB 45.7 MB/s eta 0:00:00


In [ ]:
!pip install -q "datasets==2.19.1"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 542.0/542.0 kB 10.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 172.0/172.0 kB 17.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2024.3.1 which is incompatible.


In [ ]:
import datasets
print("datasets version:", datasets.__version__)

datasets version: 2.19.1


In [ ]:
import torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
)

In [ ]:
def load_4bit_model(model_id: str):
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
    )

    tokenizer = AutoTokenizer.from_pretrained(model_id)

    if tokenizer.pad_token_id is None:
        tokenizer.pad_token_id = tokenizer.eos_token_id

    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        quantization_config=bnb_config,
        device_map="auto",
    )
    model.eval()

    print(f"{model_id} loaded in 4-bit.")
    print("pad_token_id:", tokenizer.pad_token_id, "eos_token_id:", tokenizer.eos_token_id)
    return tokenizer, model

In [ ]:
llama_id   = "meta-llama/Llama-3.1-8B-Instruct"
qwen7_id   = "Qwen/Qwen2.5-7B-Instruct"
qwen14_id  = "Qwen/Qwen2.5-14B-Instruct"

llama_tok, llama_model   = load_4bit_model(llama_id)
q7_tok, q7_model         = load_4bit_model(qwen7_id)
q14_tok, q14_model       = load_4bit_model(qwen14_id)

tokenizer_config.json:   0%|          | 0.00/55.4k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

...

model-00006-of-00008.safetensors:   0%|          | 0.00/4.00G [00:00<?, ?B/s]

model-00007-of-00008.safetensors:   0%|          | 0.00/4.00G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/8 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Qwen/Qwen2.5-14B-Instruct loaded in 4-bit.
pad_token_id: 151643 eos_token_id: 151645


In [ ]:
entail_ds = load_dataset("persiannlp/parsinlu_entailment")
print(entail_ds)
print(entail_ds["train"][0])
print(entail_ds["train"].unique("label"))

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Generating train split:   0%|          | 0/755 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1675 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/270 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['sent1', 'sent2', 'category', 'label'],
        num_rows: 755
    })
    test: Dataset({
        features: ['sent1', 'sent2', 'category', 'label'],
        num_rows: 1675
    })
    validation: Dataset({
        features: ['sent1', 'sent2', 'category', 'label'],
        num_rows: 270
    })
})
{'sent1': 'زنان به قدری بخش بزرگی از نیروی کار را تشکیل می دهند که به سختی می توان باور داشت که اگر این امر در مورد زنان  صادق نباشد ، این امر می تواند صادق باشد.', 'sent2': 'مردان بخش عظیمی از نیروی کار هستند بنابراین تنها افراد مهم هستند.', 'category': 'translation-train', 'label': 'c'}
['c', 'n', 'e', 'xx']


In [ ]:
def build_entailment_prompt(ex):
    s1 = ex["sent1"]
    s2 = ex["sent2"]
    prompt = f"""جمله ۱: «{s1}»
جمله ۲: «{s2}»

رابطهٔ منطقی بین جمله ۱ و جمله ۲ چیست؟
فقط یکی از این حروف را به عنوان پاسخ بنویس (بدون هیچ توضیح دیگر):
e  (اگر جمله ۲ از جمله ۱ نتیجه می‌شود)
c  (اگر جمله ۲ با جمله ۱ تناقض دارد)
n  (اگر نه نتیجه است و نه تناقض و خنثی است)

پاسخ:"""
    return prompt

In [ ]:
@torch.no_grad()
def generate_label(model, tokenizer, prompt, max_new_tokens=4):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    out = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        num_beams=1,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )

    full = tokenizer.decode(out[0], skip_special_tokens=True)
    answer = full[len(prompt):].strip().lower()

    if not answer:
        return None
    first = answer[0]
    return first if first in ["e", "c", "n"] else None

In [ ]:
def eval_parsinlu_entailment(model, tokenizer, dataset, split="test", max_samples=None):
    data = dataset[split]
    correct = 0
    total = 0

    for i, ex in enumerate(data):
        if max_samples is not None and i >= max_samples:
            break

        gold = ex["label"]

        # skip uncertain labels
        if gold == "xx":
            continue

        prompt = build_entailment_prompt(ex)
        pred = generate_label(model, tokenizer, prompt)

        if pred is None:
            continue

        total += 1
        if pred == gold:
            correct += 1

        if (i + 1) % 100 == 0:
            print(f"{i+1} examples processed… current acc = {correct / max(total,1):.3f}")

    acc = correct / total if total > 0 else 0.0
    print(f"Used {total} examples (skipped xx / unparsable).")
    print(f"Accuracy on {split}: {acc:.4f}")
    return acc

In [ ]:
llama_entail_100 = eval_parsinlu_entailment(llama_model, llama_tok, entail_ds, split="test", max_samples=100)
q7_entail_100    = eval_parsinlu_entailment(q7_model,   q7_tok,   entail_ds, split="test", max_samples=100)
q14_entail_100   = eval_parsinlu_entailment(q14_model,  q14_tok,  entail_ds, split="test", max_samples=100)

The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


100 examples processed… current acc = 0.530
Used 100 examples (skipped xx / unparsable).
Accuracy on test: 0.5300
100 examples processed… current acc = 0.610
Used 100 examples (skipped xx / unparsable).
Accuracy on test: 0.6100
100 examples processed… current acc = 0.680
Used 100 examples (skipped xx / unparsable).
Accuracy on test: 0.6800


In [ ]:
llama_entail_full = eval_parsinlu_entailment(llama_model, llama_tok, entail_ds, split="test")
q7_entail_full    = eval_parsinlu_entailment(q7_model,   q7_tok,   entail_ds, split="test")
q14_entail_full   = eval_parsinlu_entailment(q14_model,  q14_tok,  entail_ds, split="test")

100 examples processed… current acc = 0.530
200 examples processed… current acc = 0.560
300 examples processed… current acc = 0.543
400 examples processed… current acc = 0.550
500 examples processed… current acc = 0.530
600 examples processed… current acc = 0.535
700 examples processed… current acc = 0.550
800 examples processed… current acc = 0.544
900 examples processed… current acc = 0.546
1000 examples processed… current acc = 0.552
1100 examples processed… current acc = 0.546
1200 examples processed… current acc = 0.539
1300 examples processed… current acc = 0.535
1400 examples processed… current acc = 0.536
1500 examples processed… current acc = 0.537
1600 examples processed… current acc = 0.538
Used 1674 examples (skipped xx / unparsable).
Accuracy on test: 0.5376
100 examples processed… current acc = 0.610
200 examples processed… current acc = 0.620
300 examples processed… current acc = 0.633
400 examples processed… current acc = 0.618
500 examples processed… current acc = 0.61

# Llama 3.1 8B accuracy on Parsinlu_entailment -> 53.76%
# Qwen 2.5 7B accuracy on Parsinlu_entailment -> 61.19%
# Qwen 2.5 14B accuracy on Parsinlu_entailment -> 62.69%

# Parsinlu_sentiment

In [ ]:
sent_ds = load_dataset("persiannlp/parsinlu_sentiment")
print(sent_ds)
print(sent_ds["train"][0])
print(sent_ds["train"].unique("label"))

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Generating train split:   0%|          | 0/13617 [00:00<?, ? examples/s]

Generating test_food split:   0%|          | 0/1344 [00:00<?, ? examples/s]

Generating test_movies split:   0%|          | 0/816 [00:00<?, ? examples/s]

Generating validation_food split:   0%|          | 0/1330 [00:00<?, ? examples/s]

Generating validation_movies split:   0%|          | 0/360 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['review', 'review_id', 'example_id', 'excel_id', 'question', 'category', 'aspect', 'label', 'guid'],
        num_rows: 13617
    })
    test_food: Dataset({
        features: ['review', 'review_id', 'example_id', 'excel_id', 'question', 'category', 'aspect', 'label', 'guid'],
        num_rows: 1344
    })
    test_movies: Dataset({
        features: ['review', 'review_id', 'example_id', 'excel_id', 'question', 'category', 'aspect', 'label', 'guid'],
        num_rows: 816
    })
    validation_food: Dataset({
        features: ['review', 'review_id', 'example_id', 'excel_id', 'question', 'category', 'aspect', 'label', 'guid'],
        num_rows: 1330
    })
    validation_movies: Dataset({
        features: ['review', 'review_id', 'example_id', 'excel_id', 'question', 'category', 'aspect', 'label', 'guid'],
        num_rows: 360
    })
})
{'review': 'دوستان حتما دقت کنید درقسمت فیله تکه های سبز رنگ داشت که نشان دهنده انتیبیوتیک هایی ه

In [ ]:
from datasets import concatenate_datasets

test_food   = sent_ds["test_food"]
test_movies = sent_ds["test_movies"]

sent_test = concatenate_datasets([test_food, test_movies])
print(sent_test)

Dataset({
    features: ['review', 'review_id', 'example_id', 'excel_id', 'question', 'category', 'aspect', 'label', 'guid'],
    num_rows: 2160
})


In [ ]:
def build_sentiment_prompt(ex):
    review  = ex["review"]
    aspect  = ex["aspect"]
    question = ex["question"]  # optional, but useful context

    prompt = f"""جمله زیر یک نظر کاربر است.

نظر: «{review}»

سؤال: {question}
جنبه مورد نظر: «{aspect}»

امتیاز احساس (سنتیمنت) کاربر نسبت به این جنبه را تعیین کن.
فقط و فقط یکی از این اعداد را به عنوان پاسخ بنویس (بدون هیچ کلمه یا توضیح دیگری):
-3  (بسیار منفی)
-2  (منفی)
-1  (کمی منفی)
0   (خنثی)
1   (کمی مثبت)
2   (مثبت)
3   (بسیار مثبت)

پاسخ:"""
    return prompt

In [ ]:
import torch

@torch.no_grad()
def generate_raw_output(model, tokenizer, prompt, max_new_tokens=8):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    out = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        num_beams=1,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )
    full = tokenizer.decode(out[0], skip_special_tokens=True)
    return full[len(prompt):].strip()

In [ ]:
def extract_sentiment_label(output_text, allowed_labels):
    text = output_text.strip()

    if text in allowed_labels:
        return text

    for lab in allowed_labels:
        if lab in text.split():
            return lab

    for token in text.replace("،", " ").replace(",", " ").split():
        if token in allowed_labels:
            return token

    return None

In [ ]:
def eval_parsinlu_sentiment(model, tokenizer, sent_test, max_samples=None):
    labels = sorted(list(set(sent_test.unique("label"))), key=lambda x: int(x))
    print("Allowed labels:", labels)

    correct = 0
    total = 0

    for i, ex in enumerate(sent_test):
        if max_samples is not None and i >= max_samples:
            break

        gold = ex["label"]
        prompt = build_sentiment_prompt(ex)
        out_text = generate_raw_output(model, tokenizer, prompt, max_new_tokens=8)
        pred = extract_sentiment_label(out_text, labels)

        if pred is None:
            # could not parse model output, so skip
            continue

        total += 1
        if pred == gold:
            correct += 1

        if (i + 1) % 200 == 0:
            print(f"{i+1} examples… current acc = {correct / max(total,1):.3f}")

    acc = correct / total if total > 0 else 0.0
    print(f"Used {total} examples (skipped unparsable).")
    print(f"Accuracy (sentiment): {acc:.4f}")
    return acc

In [ ]:
llama_sent_200 = eval_parsinlu_sentiment(llama_model, llama_tok, sent_test, max_samples=200)
q7_sent_200    = eval_parsinlu_sentiment(q7_model,   q7_tok,   sent_test, max_samples=200)
q14_sent_200   = eval_parsinlu_sentiment(q14_model,  q14_tok,  sent_test, max_samples=200)

In [ ]:
llama_sent_full = eval_parsinlu_sentiment(llama_model, llama_tok, sent_test)
q7_sent_full    = eval_parsinlu_sentiment(q7_model,   q7_tok,   sent_test)
q14_sent_full   = eval_parsinlu_sentiment(q14_model,  q14_tok,  sent_test)

Allowed labels: ['-3', '-2', '-1', '0', '1', '2', '3']
200 examples… current acc = 0.240
400 examples… current acc = 0.215
600 examples… current acc = 0.195
800 examples… current acc = 0.190
1000 examples… current acc = 0.193
1200 examples… current acc = 0.193
1400 examples… current acc = 0.188
1800 examples… current acc = 0.178
2000 examples… current acc = 0.173
Used 2089 examples (skipped unparsable).
Accuracy (sentiment): 0.1671
Allowed labels: ['-3', '-2', '-1', '0', '1', '2', '3']
200 examples… current acc = 0.120
400 examples… current acc = 0.107
600 examples… current acc = 0.112
800 examples… current acc = 0.121
1000 examples… current acc = 0.131
1200 examples… current acc = 0.128
1400 examples… current acc = 0.130
1600 examples… current acc = 0.122
1800 examples… current acc = 0.121
2000 examples… current acc = 0.122
Used 2159 examples (skipped unparsable).
Accuracy (sentiment): 0.1232
Allowed labels: ['-3', '-2', '-1', '0', '1', '2', '3']
200 examples… current acc = 0.140
400 

# Llama 3.1 8B accuracy on Parsinlu_sentiment -> 16.71%
# Qwen 2.5 7B accuracy on Parsinlu_sentiment -> 12.32%
# Qwen 2.5 14B accuracy on Parsinlu_sentiment -> 11.16%